# SOTA CPT Training v2 — Spurgeon / Puritans / Theology (Notebook B_sota)

**Does not replace** `B_training.ipynb` (Phase-1 Spurgeon-only baseline).

Plan: `continued_pretrain/PLAN_FABLE5_TO_IMPROVE_CPT.md`

### Flagship recipe (v2)
- Base: **`unsloth/Qwen3.5-4B-Base`** (Apache 2.0). Fallback: `unsloth/Qwen2.5-3B` if M1 fails.
- QLoRA r=64 + rsLoRA; targets attn+MLP+`embed_tokens`(+`lm_head` per D4)
- **`UnslothTrainer`** dual LR: body `5e-5`, embeddings `1e-5` (fallback `5e-6` on fp16 spikes)
- `warmup_ratio=0.03`, packing @ 2048, `max_steps` multi-session resume
- Per-bucket eval dict; `load_best_model_at_end` on `eval_mix_loss`
- Diagnostics D1/D2/D4; run config records manifest SHA + pip freeze

### 9B (E3 — not flagship)
Only after VRAM probe (~20 steps, `max_memory_reserved` < ~15 GB). Full dual-LR at seq 2048
is over-budget on single T4.

### VRAM escape hatches (T4 16GB)
1. `TRAIN_LM_HEAD = False` if D4 shows tied embeddings or OOM
2. `TRAIN_EMBEDDINGS = False`
3. `LORA_RANK = 32`
4. `PER_DEVICE_BATCH = 1`, raise `GRAD_ACCUM`


## 1. Install Dependencies (G1 — pin after first good run)

In [ ]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

# Record environment lock for multi-session resume (G1)
import subprocess, pathlib
lock = pathlib.Path("/kaggle/working/requirements_lock.txt")
try:
    freeze = subprocess.check_output(["pip", "freeze"], text=True)
    lock.write_text(freeze, encoding="utf-8")
    print("Wrote", lock, "lines=", freeze.count(chr(10)))
except Exception as e:
    print("pip freeze skipped:", e)


## 2. Config (edit this cell only)

In [ ]:
import os
import json
import hashlib
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Unsloth may offload embeddings to disk when training them — must be writable (not /kaggle/input)
os.environ.setdefault("UNSLOTH_COMPILE_DISABLE", "0")
OFFLOAD_DIR = "/kaggle/working/unsloth_offload"
os.makedirs(OFFLOAD_DIR, exist_ok=True)
os.environ["HF_HOME"] = "/kaggle/working/hf_home"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

# ---- Model / LoRA (flagship §4.1 + M1 gate) ----
# M1 risk: Qwen3.5 is hybrid (linear_attention + full_attention + vision_config).
# If from_pretrained / train fails on T4, set MODEL_NAME to fallback immediately.
MODEL_NAME = "unsloth/Qwen3.5-4B-Base"  # M1 fail → "unsloth/Qwen2.5-3B"
# MODEL_NAME = "unsloth/Qwen3.5-9B-Base"  # E3 only after VRAM probe passes
MAX_SEQ_LENGTH = 2048
LORA_RANK = 64
LORA_ALPHA = 64
USE_RSLORA = True
TRAIN_EMBEDDINGS = True
# M1 (2026-07-13): Qwen3.5-4B-Base has tie_word_embeddings=true → default lm_head OFF.
# Flip to True only if D4 shows clean separate head training + VRAM holds.
TRAIN_LM_HEAD = False
LORA_DROPOUT = 0

# ---- Dual LR (Unsloth CPT; v2 emb LR) ----
LEARNING_RATE = 5e-5
EMBEDDING_LEARNING_RATE = 1e-5  # fallback 5e-6 on fp16 spikes
WARMUP_RATIO = 0.03             # v2: was fixed warmup_steps=100
WEIGHT_DECAY = 0.01
LR_SCHEDULER = "cosine"

# ---- Batch / steps ----
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 8
# D3 (2026-07-13): ~8.2M train tokens / 32768 tok/step ≈ 250 steps/epoch.
# Reconfirm with D1 packed tokens_per_epoch on Kaggle, then lock for multi-session resume.
MAX_STEPS = 250
NUM_TRAIN_EPOCHS = 1.0         # ignored when MAX_STEPS is set
LOGGING_STEPS = 10
EVAL_STEPS = 50                # shorter epoch → more frequent eval
SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = 2           # v2: was 3
LOAD_BEST_MODEL_AT_END = True
METRIC_FOR_BEST = "eval_mix_loss"
REPORT_TO = "none"             # or "wandb" with Kaggle secret

# ---- Paths (Kaggle) ----
SRC_DATASET_PATH = "/kaggle/input/datasets/rafaelvieira1/theology-cpt-dataset/theology_dataset"
SRC_HOLDOUT_PATH = "/kaggle/input/datasets/rafaelvieira1/theology-cpt-dataset/theology_holdouts"
LOCAL_DATASET_PATH = "/kaggle/working/theology_dataset"
LOCAL_HOLDOUT_PATH = "/kaggle/working/theology_holdouts"
OUTPUT_DIR = "/kaggle/working/checkpoints_sota"
ADAPTER_OUT = "/kaggle/working/theology_cpt_lora"
RUN_CONFIG_OUT = "/kaggle/working/theology_cpt_run_config.json"
MANIFEST_PATH = "/kaggle/input/datasets/rafaelvieira1/theology-cpt-corpus/theology_mix_manifest.json"
# Optional local/corpus mount for manifest hash:
# MANIFEST_PATH = "/kaggle/input/datasets/rafaelvieira1/theology-cpt-corpus/theology_mix_manifest.json"

PREV_RUN_CHECKPOINT = None
SEED = 42
APPEND_EOS = True              # D2 fix if packed rows lack EOS
EVAL_DOCS_PER_BUCKET = 8

print("Config ready.")
print(f"  model={MODEL_NAME} seq={MAX_SEQ_LENGTH} r={LORA_RANK} rslora={USE_RSLORA}")
print(f"  lr={LEARNING_RATE} emb_lr={EMBEDDING_LEARNING_RATE} warmup_ratio={WARMUP_RATIO}")
print(f"  train_embed={TRAIN_EMBEDDINGS} train_lm_head={TRAIN_LM_HEAD}")
print(f"  offload_dir={OFFLOAD_DIR}")

## 3. Model & PEFT (CPT targets) + D4 tied-embeddings check

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
if TRAIN_LM_HEAD:
    target_modules.append("lm_head")
if TRAIN_EMBEDDINGS:
    target_modules.append("embed_tokens")

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=target_modules,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=USE_RSLORA,
)

# ---- D4: tied embeddings ----
d4 = {
    "tie_word_embeddings": getattr(model.config, "tie_word_embeddings", None),
    "same_storage": None,
    "trainable_embed_or_head": [],
}
try:
    emb = model.get_input_embeddings().weight
    head = model.get_output_embeddings().weight
    d4["same_storage"] = int(emb.data_ptr() == head.data_ptr())
except Exception as e:
    d4["error"] = str(e)
for n, p in model.named_parameters():
    if ("embed_tokens" in n or "lm_head" in n) and p.requires_grad:
        d4["trainable_embed_or_head"].append({"name": n, "shape": list(p.shape)})

print("D4 tied embeddings:", json.dumps(d4, indent=2))
if d4.get("same_storage") and TRAIN_LM_HEAD:
    print(
        "WARNING: base weights share storage (tied). "
        "If lm_head is not truly trainable separately, set TRAIN_LM_HEAD=False and re-run PEFT cell."
    )
print("Target modules:", target_modules)
try:
    model.print_trainable_parameters()
except Exception:
    pass

## 4. Dataset + per-bucket eval + UnslothTrainer

In [ ]:
from unsloth import UnslothTrainer, UnslothTrainingArguments
from datasets import load_from_disk
import shutil

if not os.path.exists(LOCAL_DATASET_PATH):
    if not os.path.exists(SRC_DATASET_PATH):
        raise FileNotFoundError(
            f"Dataset not found at {SRC_DATASET_PATH}. "
            "Run A_data_prep_sota.ipynb and mount theology-cpt-dataset."
        )
    print(f"Copying dataset {SRC_DATASET_PATH} -> {LOCAL_DATASET_PATH} ...")
    shutil.copytree(SRC_DATASET_PATH, LOCAL_DATASET_PATH)
else:
    print(f"Using writable dataset at {LOCAL_DATASET_PATH}")

dataset = load_from_disk(LOCAL_DATASET_PATH)
print(dataset)

# Ensure EOS on each doc (D2) so packing preserves document boundaries
train_ds = dataset["train"]
if APPEND_EOS and tokenizer.eos_token:
    def _add_eos(batch):
        eos = tokenizer.eos_token
        texts = []
        for t in batch["text"]:
            t = t if t.endswith(eos) else (t + eos)
            texts.append(t)
        return {"text": texts}
    train_ds = train_ds.map(_add_eos, batched=True, desc="append EOS")

# Per-bucket eval dict (v2)
eval_sets = {}
mix_eval = dataset.get("test") or dataset.get("validation")
if mix_eval is not None:
    eval_sets["mix"] = mix_eval.select(range(min(EVAL_DOCS_PER_BUCKET * 2, len(mix_eval))))

holdout_src = SRC_HOLDOUT_PATH if os.path.exists(SRC_HOLDOUT_PATH) else LOCAL_HOLDOUT_PATH
if os.path.exists(holdout_src):
    if holdout_src != LOCAL_HOLDOUT_PATH and not os.path.exists(LOCAL_HOLDOUT_PATH):
        shutil.copytree(holdout_src, LOCAL_HOLDOUT_PATH)
        holdout_src = LOCAL_HOLDOUT_PATH
    for name in ["spurgeon", "puritan", "confession", "general"]:
        p = os.path.join(holdout_src, name)
        if os.path.exists(p):
            ds = load_from_disk(p)
            if APPEND_EOS and tokenizer.eos_token and "text" in ds.column_names:
                eos = tokenizer.eos_token
                ds = ds.map(
                    lambda batch: {
                        "text": [t if t.endswith(eos) else t + eos for t in batch["text"]]
                    },
                    batched=True,
                )
            eval_sets[name] = ds.select(range(min(EVAL_DOCS_PER_BUCKET, len(ds))))
            print(f"  eval[{name}]={len(eval_sets[name])}")
else:
    print("NOTE: no multi-holdouts found; eval will use mix split only.")

if not eval_sets and mix_eval is not None:
    eval_sets = mix_eval

training_args = UnslothTrainingArguments(
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=WARMUP_RATIO,
    learning_rate=LEARNING_RATE,
    embedding_learning_rate=EMBEDDING_LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=LOAD_BEST_MODEL_AT_END and isinstance(eval_sets, dict) and "mix" in eval_sets,
    metric_for_best_model=METRIC_FOR_BEST if LOAD_BEST_MODEL_AT_END else None,
    greater_is_better=False,
    output_dir=OUTPUT_DIR,
    seed=SEED,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    report_to=REPORT_TO,
)

if MAX_STEPS is not None:
    training_args.max_steps = int(MAX_STEPS)
else:
    training_args.num_train_epochs = float(NUM_TRAIN_EPOCHS)

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_sets if eval_sets else None,
    args=training_args,
)

print("Trainer ready:", type(trainer).__name__)
print(f"  train size={len(train_ds)}")
print(f"  eval keys={list(eval_sets) if isinstance(eval_sets, dict) else type(eval_sets)}")

## 5. Diagnostics D1 (truncation) + D2 (EOS)

In [ ]:
import numpy as np

# D1 — where did my tokens go?
tds = trainer.train_dataset
n = len(tds)
# Packed datasets may expose input_ids only after collator; try both
def _row_len(i):
    row = tds[i]
    if "input_ids" in row:
        return len(row["input_ids"])
    if "text" in row:
        return len(tokenizer(row["text"], add_special_tokens=False)["input_ids"])
    return -1

step = max(1, n // 200)
lens = [_row_len(i) for i in range(0, n, step)]
lens = [x for x in lens if x > 0]
d1 = {
    "packed_or_raw_rows": n,
    "sampled": len(lens),
    "row_token_len_min": int(min(lens)) if lens else None,
    "row_token_len_p50": int(np.median(lens)) if lens else None,
    "row_token_len_max": int(max(lens)) if lens else None,
    "tokens_per_epoch_est": int(n * float(np.mean(lens))) if lens else None,
}
print("D1 truncation diagnostic:", json.dumps(d1, indent=2))
if d1["row_token_len_max"] and d1["row_token_len_max"] <= MAX_SEQ_LENGTH and n < 5000:
    print(
        "NOTE: If tokens_per_epoch_est << corpus tokens and max≈2048 with ~1 row/doc, "
        "truncation is still present — rebuild mix with max_chunk_chars=7000."
    )

# D2 — EOS boundaries (first 5 rows)
eos = tokenizer.eos_token_id
d2_counts = []
for i in range(min(5, n)):
    row = tds[i]
    if "input_ids" in row:
        ids = list(row["input_ids"])
    else:
        ids = tokenizer(row["text"], add_special_tokens=False)["input_ids"]
    d2_counts.append(ids.count(eos))
print("D2 EOS counts (first 5):", d2_counts)
if d2_counts and max(d2_counts) == 0:
    print("WARNING: no EOS found — ensure APPEND_EOS=True and re-run dataset cell.")

## 6. Optional: 9B VRAM probe (E3 gate — skip for 4B flagship)

In [ ]:
# Run only when MODEL_NAME is the 9B experiment. Pass if peak reserved < ~15 GB.
RUN_VRAM_PROBE = False  # set True for E3
PROBE_STEPS = 20
VRAM_PROBE_LIMIT_GB = 15.0

if RUN_VRAM_PROBE:
    import time
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    t0 = time.time()
    # Short train run
    old_max = getattr(trainer.args, "max_steps", None)
    trainer.args.max_steps = PROBE_STEPS
    trainer.train()
    peak_gb = torch.cuda.max_memory_reserved() / (1024 ** 3)
    dt = time.time() - t0
    print(f"VRAM probe: peak_reserved={peak_gb:.2f} GB over {PROBE_STEPS} steps in {dt:.1f}s")
    if peak_gb < VRAM_PROBE_LIMIT_GB:
        print("PASS — multi-session 9B may proceed with this locked config.")
    else:
        print("FAIL — stay on 4B flagship or re-probe with concessions (embed-only / seq 1024 / batch 1).")
    # Do not continue long training from probe state without restart
else:
    print("VRAM probe skipped (flagship 4B path).")

## 7. Train

In [ ]:
import sys
import trl
import time

# Pickle guard (Phase-1 / v1)
if hasattr(trainer, "args"):
    cls_name = trainer.args.__class__.__name__
    if cls_name in ("SFTConfig", "UnslothTrainingArguments"):
        try:
            import trl.trainer.sft_config as sft_config_mod
            sft_config_mod.SFTConfig = trainer.args.__class__
            sys.modules["trl.trainer.sft_config"].SFTConfig = trainer.args.__class__
            trl.SFTConfig = trainer.args.__class__
        except Exception as e:
            print("Pickle guard skipped:", e)

# Measure s/step over first steps if MAX_STEPS not set (guidance for multi-session)
print("Starting SOTA CPT v2...")
t0 = time.time()
if PREV_RUN_CHECKPOINT:
    if not os.path.exists(PREV_RUN_CHECKPOINT):
        raise FileNotFoundError(f"Checkpoint not found: {PREV_RUN_CHECKPOINT}")
    print(f"Resuming from {PREV_RUN_CHECKPOINT}")
    train_result = trainer.train(resume_from_checkpoint=PREV_RUN_CHECKPOINT)
else:
    train_result = trainer.train()
elapsed = time.time() - t0
print(train_result)
print(f"Wall time: {elapsed/3600:.2f} h")

## 8. Save adapter + run config (manifest hash, D1–D4, freeze)

In [ ]:
def _sha256_file(p):
    h = hashlib.sha256()
    try:
        with open(p, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return None

manifest_meta = None
if os.path.exists(MANIFEST_PATH):
    try:
        with open(MANIFEST_PATH, encoding="utf-8") as f:
            manifest_meta = json.load(f)
    except Exception as e:
        print("manifest load failed:", e)

run_config = {
    "plan": "PLAN_FABLE5_TO_IMPROVE_CPT v2",
    "model_name": MODEL_NAME,
    "flagship": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "use_rslora": USE_RSLORA,
    "target_modules": target_modules,
    "learning_rate": LEARNING_RATE,
    "embedding_learning_rate": EMBEDDING_LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "per_device_batch": PER_DEVICE_BATCH,
    "grad_accum": GRAD_ACCUM,
    "max_steps": MAX_STEPS,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "seed": SEED,
    "dataset_src": SRC_DATASET_PATH,
    "prev_checkpoint": PREV_RUN_CHECKPOINT,
    "manifest_path": MANIFEST_PATH,
    "manifest_sha256": _sha256_file(MANIFEST_PATH) if os.path.exists(MANIFEST_PATH) else None,
    "manifest_created_at": (manifest_meta or {}).get("created_at"),
    "manifest_buckets": (manifest_meta or {}).get("buckets"),
    "d4_tied_embeddings": d4,
    "d1_truncation": d1 if "d1" in dir() else None,
    "d2_eos_counts": d2_counts if "d2_counts" in dir() else None,
    "requirements_lock": "/kaggle/working/requirements_lock.txt",
    "offload_dir": OFFLOAD_DIR,
    "notes": "SOTA CPT v2 — dual LR, per-bucket eval, Qwen3.5-4B flagship. Baseline B_training.ipynb frozen.",
}

print(f"Saving adapter to {ADAPTER_OUT} ...")
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)

with open(RUN_CONFIG_OUT, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

print("Saved:")
print(" ", ADAPTER_OUT)
print(" ", RUN_CONFIG_OUT)
print("Checkpoints:", OUTPUT_DIR)